# Transformer


Transformer 모델은 자연어 처리와 같은 시퀀스 데이터를 처리하는 데 혁신적인 방법을 제시한 딥러닝 모델이다. 기존 RNN 기반 모델의 한계를 극복하고 효율성과 성능을 크게 향상시킨 모델로, "Attention is All You Need" 논문(2017)에서 처음 소개되었다.

<table>
<tr>
  <th colspan=1>The original Transformer diagram</th>
  <th colspan=1>A representation of a 4-layer Transformer</th>
</tr>
<tr>
  <td>
   <img width=400 src="https://www.tensorflow.org/images/tutorials/transformer/transformer.png"/>
  </td>
  <td>
   <img width=307 src="https://www.tensorflow.org/images/tutorials/transformer/Transformer-4layer-compact.png"/>
  </td>
</tr>
</table>

<br/>

<img src="https://www.tensorflow.org/images/tutorials/transformer/apply_the_transformer_to_machine_translation.gif" alt="Applying the Transformer to machine translation">

![https://www.tensorflow.org/text/tutorials/transformer#scaled_dot_product_attention](https://d.pr/i/O6WlEu+)



**Transformer 모델의 주요 구성 요소**
Transformer는 크게 두 가지 블록으로 구성된다: **인코더(Encoder)** 와 **디코더(Decoder)**. 인코더는 입력 시퀀스를 처리하고, 디코더는 이를 기반으로 출력 시퀀스를 생성한다.

1. **Self-Attention Mechanism**
   - 입력 데이터 내의 각 단어가 다른 단어와 어떤 관계를 가지는지 학습하는 메커니즘이다.
   - 단어 간 상관관계를 효율적으로 계산하며, 이를 통해 문맥 정보를 반영한다.
   - 핵심 구성 요소:
     - **Query (Q)**: 현재 단어의 정보
     - **Key (K)**: 비교 대상 단어의 정보
     - **Value (V)**: 실제 반환될 정보
     $$ Attention(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V $$

2. **Multi-Head Attention**
   - Self-Attention을 여러 개의 서로 다른 공간에서 병렬적으로 계산하여 다양한 문맥 정보를 학습한다.
   - 각 헤드에서 서로 다른 부분의 관계를 포착하고 이를 결합한다.

3. **Feed-Forward Neural Network**
   - 각 단어 위치별로 독립적으로 처리하는 완전연결층.
   - 비선형성을 추가하여 모델의 표현력을 높인다.

4. **Positional Encoding**
   - Transformer는 RNN처럼 순차적인 정보 흐름을 따르지 않으므로, 단어의 위치 정보를 추가적으로 학습해야 한다.
   - 이를 위해 각 단어에 고유한 위치 정보를 더하는 Positional Encoding을 사용한다.
   - 일반적으로 사인(sin)과 코사인(cos) 함수를 이용해 계산된다.

5. **Residual Connection**
   - 학습이 더 잘 이루어지도록, 각 층의 입력을 출력에 더하는 연결을 사용한다.
   - 이로 인해 모델이 더 깊어져도 학습이 안정적으로 진행된다.

6. **Layer Normalization**
   - 각 층의 출력을 정규화하여 학습 속도를 높이고 안정성을 확보한다.

## 1. Transformer Hyperparameter

 트랜스포머를 제안한 논문에서 사용한 수치로 하이퍼파라미터는 사용자가 모델 설계시 임의로 변경할 수 있는 값들이다.

- **$d_{model} = 512$**

  $d_{model}$은 트랜스포머의 인코더와 디코더에서 입력과 출력의 크기를 의미한다. 또한, 임베딩 벡터의 차원 역시 $d_{model}$이며, 각 인코더와 디코더가 다음 층으로 값을 전달할 때도 이 차원을 유지한다. 논문에서는 $d_{model}$을 512로 설정하였다.

- **$num\_layers = 6$**

  $num\_layers$는 트랜스포머에서 인코더와 디코더가 각각 몇 층으로 구성되었는지를 나타낸다. 하나의 인코더 또는 디코더를 하나의 층으로 간주하며, 논문에서는 인코더와 디코더를 각각 6층으로 쌓았다.

- **$num\_heads = 8$**

  $num\_heads$는 트랜스포머에서 멀티 헤드 어텐션의 병렬 개수를 의미한다. 트랜스포머는 어텐션을 한 번 수행하는 대신 여러 개로 분할하여 병렬로 수행한 뒤, 결과값을 다시 하나로 합친다. 논문에서는 $num\_heads$를 8로 설정하였다.

- **$d_{ff} = 2048$**

  $d_{ff}$는 트랜스포머 내부 피드 포워드 신경망의 은닉층 크기를 의미한다. 피드 포워드 신경망의 입력층과 출력층 크기는 $d_{model}$과 동일하며, 논문에서는 은닉층 크기를 2048로 설정하였다.

- **$dropout = 0.1$**

  $dropout$은 드롭아웃을 적용할 확률을 나타낸다. 트랜스포머 모델에서 드롭아웃을 사용하였으며, 논문에서는 드롭아웃 비율을 0.1로 설정하였다.


In [73]:
# 논문에서 제시된 하이퍼 파라미터 선언
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

d_model = 512                   # 토큰 하나를 512차원 벡터로 표현
num_layers = 6                  # encoder, decoder layer를 6개씩 쌓는다.
num_heads= 8                    # multi-head attention head 8개로 설정
d_k = d_model // num_heads      # 분할된 Q, K, V의 차원 수 (64*8)
d_ff = 2048                     # feed forward layer : 512 -> 2048 -> 512
drop_out_rate = 0.1

# 원문 사전 크기
src_vocab_size = 10000
# 번역문 사전 크기
trg_vocab_size = 10000

batch_size = 64
seq_len = 128

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## transformer 구성 요소

In [74]:
class PositionalEncoding(nn.Module):
    pass

class Encoder(nn.Module):
    pass

class Decoder(nn.Module):
    pass

class Transformer(nn.Module):
    def __init__(self, src_vocab_size,trg_vocab_size,d_model):
        super().__init__()
        # 원문 입력 토큰과 번역문 토큰 id를 벡터로 변환한다.
        self.src_embedding = nn.Embedding(src_vocab_size,d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size,d_model)
        # Transformer는 RNN처럼 앞에서 뒤로 순서대로 읽지 않는다.
        # 따라서 단어 벡터만 넣으면 모델은 토큰의 순서를 모르기 때문에 각 토큰 벡터의 위치정보를 더한다.
        self.positional_encoding = PositionalEncoding()
        # encoder는 입력 문장을 문맥 벡터로 변환하고, Decoder는 그 결과를 참고해 출력 문자을 만든다.
        self.encoder = Encoder()
        self.decoder = Decoder()
        # 512차원을 번역문 사전 크기만큼 바꾼다.
        self.output_layer = nn.Linear(d_model, trg_vocab_size)
        # 마지막 차원에 대해 확률로 변경한다.
        self.softmax = nn.LogSoftmax(dim=-1)

    def forward(self, src_inputs, trg_inputs, e_mask=None, d_mask=None):
        # 원문, 번역문 토큰 id를 벡터로 변경하고, 위치 정보를 더한다.
        src_inputs = self.src_embedding(src_inputs)
        src_inputs = self.positional_encoding(src_inputs)
        trg_inputs = self.trg_embedding(trg_inputs)
        trg_inputs = self.positional_encoding(trg_inputs)

        # Encoder가 원문 전체를 읽고 문맥 정보를 만든다.
        encoder_outputs = self.encoder(src_inputs,e_mask)

        # 지금까지 생성된 번역문, encoder가 읽어둔 원문 정보를 통해 결과를 만든다.
        decoder_outputs = self.decoder(trg_inputs,encoder_outputs,e_mask,d_mask)

        # output layer
        outputs = self.output_layer(decoder_outputs)
        outputs = self.softmax(outputs)
        return outputs

### PositionalEncoding

인코더와 디코더에 대한 입력은 동일한 임베딩 및 위치 인코딩 logic을 사용한다.

<table>
<tr>
  <th colspan=1>The embedding and positional encoding layer</th>
<tr>
<tr>
  <td>
   <img src="https://www.tensorflow.org/images/tutorials/transformer/PositionalEmbedding.png"/>
  </td>
</tr>
</table>

일련의 토큰이 주어지면 입력 토큰과 대상 토큰을 `nn.Embedding` 레이어를 사용하여 벡터로 변환해야 한다.

모델 전체에서 사용되는 Attention 레이어는 입력을 순서가 없는 벡터 집합으로 간주한다. 모델에는 순환 레이어나 컨벌루션 레이어가 포함되어 있지 않기 때문에 단어 순서를 식별할 수 있는 방법이 필요하다.

Transformer는 임베딩 벡터에 "위치 인코딩"을 추가한다. 위치 인코딩은 서로 다른 주파수의 사인과 코사인 세트를 사용하여 시퀀스 전반에 걸쳐 위치 정보를 부여해서 단어의 상대적인 위치 정보를 지정할 수 있다.

논문에서는 위치 인코딩을 계산하기 위해 다음 공식을 사용하였다.

$$\Large{PE_{(pos, 2i)} = \sin(pos / 10000^{2i / d_{model}})} $$
$$\Large{PE_{(pos, 2i+1)} = \cos(pos / 10000^{2i / d_{model}})} $$

In [75]:
import math

class PositionalEncoding(nn.Module):
    def __init__(self, seq_len,d_model):
        super().__init__()
        # 위치 인코딩 값을 담을 2차원 텐서 (seq_len, d_model) = (128, 512)
        pos_encoding = torch.zeros(seq_len,d_model)

        # 모든 위치 pos와 모든 차원 i에 대해 값을 계산
        for pos in range(seq_len):
            for i in range(0,d_model,2):
                pos_encoding[pos,i] = math.sin(pos/(10000 ** (i/d_model)))
                pos_encoding[pos,i+1] = math.cos(pos/(10000 ** (i/d_model)))
        # 위치 마다 서로 다른 패턴의 값이 만들어 진다.

        pos_encoding = pos_encoding.unsqueeze(0)    # (seq_len, d_model) -> (batch_size,seq_len,d_model)

        # 위치 인코딩 같은 학습은 값이 아니라 수식으로 미리 계산되는 값이다. (학습되지 않도록 설정)
        self.pos_encoding = pos_encoding.to(device)
        self.pos_encoding.requires_grad = False

    def forward(self, x):
        # embedding 값 스케일링 (위치 인코딩이 단어 임베딩 값보다 영향이 과하거나 덜해지지 않도록)
        x = x * math.sqrt(d_model)
        x = x + self.pos_encoding[:,:x.size(1),:]       # (batch_size, deq_len, d_model)
        return x

### Position-wise Feed-Forward Network

인코더와 디코더의 각 서브-레이어(sub-layer)에는 Multi-Head Attention 이후에 Position-wise Feed-Forward Network가 적용된다. 이는 두 개의 선형 변환과 ReLU 활성화 함수로 구성된 간단한 신경망이다. 각 단어의 위치(position)마다 독립적으로 동일한 연산을 적용한다.

<table>
<tr>
  <th colspan=1>The feed forward network</th>
<tr>
<tr>
  <td>
   <img src="https://www.tensorflow.org/images/tutorials/transformer/FeedForward.png"/>
  </td>
</tr>
</table>

네트워크는 중간에 ReLU 활성화가 있는 두 개의 선형 레이어(`nn.Linear`)와 드롭아웃(`nn.Dropout`) 레이어로 구성된다. Attention 레이어와 마찬가지로 여기 코드에도 잔차 연결 및 정규화도 포함된다.

In [76]:
# feedforward에서는 토큰 간 정보를 섞지 않는다. 각 위치의 토큰 벡터와 같은 linear 변환을 독립적으로 적용한다.
class FeedForwardLayer(nn.Module):
    def __init__(self, d_model, d_ff, drop_out_rate):
        super().__init__()
        self.linear1 = nn.Linear(d_model,d_ff)
        self.relu = nn.ReLU()
        self.dropout= nn.Dropout(drop_out_rate)
        self.linear2 = nn.Linear(d_ff,d_model)

    def forward(self,x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.linear2(x)
        return x

### Layer Normalization

Layer Normalization은 각 레이어의 입력에 대해 정규화를 수행하여 학습을 안정시키고 속도를 향상시킨다. 각 샘플의 특성(feature)에 대해 평균과 분산을 계산하여 정규화한다.

<table>
<tr>
  <th colspan=2>Add and normalize</th>
<tr>
<tr>
  <td>
   <img src="https://www.tensorflow.org/images/tutorials/transformer/Add+Norm.png"/>
  </td>
</tr>
</table>

In [77]:
class LayerNormalization(nn.Module):
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        # 각 토큰 벡터의 feature를 기준으로 평균과 분산을 맞춘다.
        # elementwise_affine : 각 요소마다 다른 스케일/오프셋 값을 각기 적용한다.
        # eps : 내부 분산 계산시 0 나누기 방지 앱실론
        self.layer = nn.LayerNorm([d_model], elementwise_affine=True,eps=eps)

    def forward(self, x):
        return self.layer(x)

### Multihead Attention


<table>
<tr>
  <th colspan=1>The base attention layer</th>
</tr>
<tr>
  <td>
   <img width=430 src="https://www.tensorflow.org/images/tutorials/transformer/BaseAttention-new.png"/>
  </td>
</tr>
</table>


1. **query sequence**: 처리 중인 시퀀스로, attention을 수행하는 시퀀스이다.
2. **context sequence**: attention의 대상이 되는 시퀀스이다.

출력은 쿼리 시퀀스와 동일한 모양을 갖는다.

이 작업은 일반적인 dictionary 조회와 유사하다고 볼 수 있다. 이 조회는 "퍼지(fuzzy)"하고 "미분 가능(differentiable)"하며 "벡터화(vectorized)"된 형태로 이루어진다.  
- **퍼지**는 모호하거나 불확실한 정보를 처리할 수 있다는 것을 의미한다.  
- **미분 가능**은 연산이 그래디언트를 통해 최적화 과정에서 학습될 수 있음을 의미한다.  
- **벡터화**는 연산이 전체 시퀀스에 대해 동시에 수행되며, 각 요소가 벡터 형태로 처리된다는 것을 의미한다.

다음은 단일 쿼리에 대해 3개의 키와 3개의 값을 가진 일반 Python 사전을 사용하는 예이다:

```python
d = {'color': 'blue', 'age': 22, 'type': 'pickup'}
result = d['color']
```

- **query**는 찾고자 하는 항목이다.
- **key**는 사전이 어떤 정보를 가지고 있는지를 나타낸다.
- **value**는 해당 정보 자체이다.

일반 dictionary에서는 '쿼리'를 조회하면 일치하는 '키'를 찾아 관련 '값'을 반환한다.  
하지만, 키가 완벽하게 일치할 필요가 없는 **모호한(fuzzy)** dictionary를 상상할 수 있다.  
예를 들어, 위의 사전에서 `d["species"]`를 검색한다면, 가장 잘 일치하는 `"pickup"`을 반환하기를 기대할 수 있다.

Attention 레이어는 이와 같은 퍼지 조회를 수행하지만, 단지 최상의 키를 찾는 것에 그치지 않는다.  
'쿼리'는 각 '키'와 얼마나 잘 일치하는지에 따라 '값'을 결합한다.

Attention 레이어에서 '쿼리', '키', '값'은 모두 벡터이다.  
Attention 레이어는 '쿼리'와 '키' 벡터를 결합하여 이들이 얼마나 잘 일치하는지를 나타내는 "어텐션 점수"를 계산한다.  
이 점수에 따라 '값'에 가중치를 적용하고, 가중치를 반영한 모든 '값'의 평균을 반환한다.

- 쿼리 시퀀스의 각 위치는 '쿼리' 벡터를 제공한다.
- 컨텍스트 시퀀스는 사전과 같은 역할을 한다.  
  컨텍스트 시퀀스의 각 위치는 '키'와 '값' 벡터를 제공한다.

입력 벡터는 직접 사용되지 않는다.  
`layers.MultiHeadAttention` 레이어는 입력 벡터를 사용하기 전에 투영(projection)을 수행하기 위해 `layers.Dense` 레이어를 포함하고 있다.

In [78]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, drop_out_rate):
        super().__init__()
        # 입력 q,k,v를 각각 query, key, value로 변환하는 linear층
        # self-attention: q,k,v는 모두 같은 입력에서 온다
        # cross-attention: q는 decoder에서 오고 k,v는 encoder 출력에서 온다
        self.w_q = nn.Linear(d_model,d_model)
        self.w_k = nn.Linear(d_model,d_model)
        self.w_v = nn.Linear(d_model,d_model)

        self.drop_out = nn.Dropout(drop_out_rate)

        self.attn_softmax = nn.Softmax(dim=-1)

        # multi head 병합용 linear 층
        self.w_o = nn.Linear(d_model, d_model)
        
    def attention(self, q,k,v,mask=None):
        attn_scores = torch.matmul(q, k.transpose(-1,-2))
        attn_scores /= math.sqrt(d_k)

        # mask 적용
        # attention에서 보면 안되는 위치를 막기 위한 값으로 mask 값이 0인 위치는 매우 작은 값으로 변경한다.
        # 매우 작은 값은 softmax 연산 이후 거의 0이 된다.
        if mask is not None:
            attn_scores = attn_scores.masked_fill(mask == 0, -1e9)

        atten_weights = self.attn_softmax(attn_scores)

        atten_weights = self.drop_out(atten_weights)

        output = torch.matmul(atten_weights, v)

        return output
    
    def forward(self, q, k, v, mask=None):
        # q와 k의 길이를 따로 변수화 한다.
        # self-attention에서는 같은 길이지만, cross-attention에서는 다른 길이를 가진다. T_q: 번역문 길이, T_k: 원문 길이
        B, T_q,_ = q.size()
        _,T_k,_ = k.size()

        q = self.w_q(q)
        k = self.w_k(k)
        v = self.w_v(v)

        # 헤드 분할
        # transpose한 결과 -> (batch_size, num_heads, seq_len, head_dim)
        q_heads = q.view(B, T_q,num_heads, d_k).transpose(1,2)
        k_heads = k.view(B, T_k,num_heads, d_k).transpose(1,2)
        v_heads = v.view(B, T_k,num_heads, d_k).transpose(1,2)

        # 어텐션 연산
        attn_values = self.attention(q_heads,k_heads,v_heads,mask)

        # 헤드 통합
        output = attn_values.transpose(1,2)
        output = output.contiguous().view(B,T_q,d_model)

        output = self.w_o(output)
        return output

# Transformer 통합

In [79]:
# Encoder
class EncoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm = LayerNormalization(d_model)
        self.multihead_attention = MultiHeadAttention(d_model,drop_out_rate)
        self.drop_out = nn.Dropout(drop_out_rate)

        self.layer_norm2 = LayerNormalization(d_model)
        self.feed_forward = FeedForwardLayer(d_model, d_ff, drop_out_rate)
        self.drop_out2 = nn.Dropout(drop_out_rate)

    def forward(self,x,e_mask=None):
        # Encoder의 self-attention
        x_1 = self.layer_norm(x)

        # q,k,v가 모두 x_1이다ㅏ. 즉 입력 문장 내부 토큰끼리 서로 참고한다.
        # 마지막에 x를 더한다. (residual connection - 잔차 연결)
        # e_mask는 Encoder의 self-attention에서 <pad> 위치를 보지 못하게 막는다.
        x = x + self.drop_out(
            self.multihead_attention(x_1,x_1,x_1,e_mask)
        )

        x_2 = self.layer_norm2(x)
        x = x + self.drop_out2(
            self.feed_forward(x_2)
        )

        return x
    
# encoderlayer를 여러개 만든다.
class Encoder(nn.Module):
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([EncoderLayer() for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x, e_mask=None):
        for layer in self.layers:
            x = layer(x,e_mask)

        return self.layer_norm(x)

In [80]:
# Decoder
class DecoderLayer(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer_norm = LayerNormalization(d_model)
        self.masked_multihead_attention = MultiHeadAttention(d_model,drop_out_rate)
        self.drop_out = nn.Dropout(drop_out_rate)

        self.layer_norm2 = LayerNormalization(d_model)
        self.multihead_cross_attention = MultiHeadAttention(d_model,drop_out_rate)
        self.drop_out2 = nn.Dropout(drop_out_rate)

        self.layer_norm3 = LayerNormalization(d_model)
        self.feed_forward = FeedForwardLayer(d_model, d_ff, drop_out_rate)
        self.drop_out3 = nn.Dropout(drop_out_rate)

    def forward(self,x, e_outputs,e_mask=None,d_mask=None):
        # Dncoder의 self-attention
        x_1 = self.layer_norm(x)
        # q,k,v가 모두 x_1이다ㅏ. 즉 입력 문장 내부 토큰끼리 서로 참고한다.
        # 마지막에 x를 더한다. (residual connection - 잔차 연결)
        # d_mask는 미래 토큰의 위치를  보지 못하게 한다.
        x = x + self.drop_out(
            self.masked_multihead_attention(x_1,x_1,x_1,mask = e_mask)
        )
        x_2 = self.layer_norm2(x)
        # decoder와 cross-ateenton
        # q = x_2 (Decoder의 현재 상태), k,v = e_outputs(encoder가 만든 원문 정보)
        x = x + self.drop_out2(
            self.masked_multihead_attention(x_2, e_outputs,e_outputs,mask = e_mask)
        )
        
        x_3 = self.layer_norm3(x)
        x = x + self.drop_out3(
            self.feed_forward(x_3)
        )
        return x
    
# decoderlayer를 여러개 만든다.
class Decoder(nn.Module):
    def __init__(self, d_model, num_layers):
        super().__init__()
        self.layers = nn.ModuleList([DecoderLayer() for _ in range(num_layers)])
        self.layer_norm = LayerNormalization(d_model)

    def forward(self, x,e_outputs,e_mask=None,d_mask=None):
        for layer in self.layers:
            x = layer(x,e_outputs,e_mask,d_mask)

        return self.layer_norm(x)

In [81]:
class Transformer(nn.Module):
    def __init__(self, src_vocab_size,trg_vocab_size,d_model,seq_len,num_layers):
        super().__init__()
        self.src_embedding = nn.Embedding(src_vocab_size,d_model)
        self.trg_embedding = nn.Embedding(trg_vocab_size,d_model)
        self.positional_encoding = PositionalEncoding(seq_len, d_model)
        self.encoder = Encoder(d_model,num_layers)
        self.decoder = Decoder(d_model,num_layers)
        self.output_layer = nn.Linear(d_model, trg_vocab_size)
        self.softmax = nn.LogSoftmax(dim=-1)

    def make_encoder_mask(self, src_inputs):
        # <pad>가 아닌 위치는 1 <pad>인 위치는 0으로 처리해서 마스킹
        e_mask = (src_inputs != 0)
        # (B,1,1,scr_seq_len)의 형태로 변경하면 (B, num_heads,src_seq_len,src_seq_len)에 broadcasting 된다.
        e_mask = e_mask.unsqueeze(1).unsqueeze(2)
        return e_mask

    def make_decoder_mask(self, trg_inputs):
        padding_mask = (trg_inputs != 0)
        padding_mask = padding_mask.unsqueeze(1).unsqueeze(2)   # (B,1,1,T)
        
        # 미래 토큰 마스킹
        T = trg_inputs.size(1)
        casual_mask = torch.tril(torch.ones((T,T),device=trg_inputs.device)).bool()
        casual_mask = casual_mask.unsqueeze(0).unsqueeze(0) # (T,T) -> (1,1,T,T)

        d_mask = padding_mask & casual_mask     # (B,1,T,T)
        return d_mask


    def forward(self, src_inputs, trg_inputs, e_mask=None, d_mask=None):

        if e_mask is None:
            e_mask = self.make_encoder_mask(src_inputs)
        if d_mask is None:
            d_mask = self.make_decoder_mask(trg_inputs)

        src_inputs = self.src_embedding(src_inputs)
        src_inputs = self.positional_encoding(src_inputs)
        trg_inputs = self.trg_embedding(trg_inputs)
        trg_inputs = self.positional_encoding(trg_inputs)

        encoder_outputs = self.encoder(src_inputs,e_mask)

        decoder_outputs = self.decoder(trg_inputs,encoder_outputs,e_mask,d_mask)

        outputs = self.output_layer(decoder_outputs)
        outputs = self.softmax(outputs)
        return outputs

In [82]:
model = Transformer(src_vocab_size,trg_vocab_size,d_model,seq_len,num_layers)
model = model.to(device)

model

Transformer(
  (src_embedding): Embedding(10000, 512)
  (trg_embedding): Embedding(10000, 512)
  (positional_encoding): PositionalEncoding()
  (encoder): Encoder(
    (layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (layer_norm): LayerNormalization(
          (layer): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
        )
        (multihead_attention): MultiHeadAttention(
          (w_q): Linear(in_features=512, out_features=512, bias=True)
          (w_k): Linear(in_features=512, out_features=512, bias=True)
          (w_v): Linear(in_features=512, out_features=512, bias=True)
          (drop_out): Dropout(p=0.1, inplace=False)
          (attn_softmax): Softmax(dim=-1)
          (w_o): Linear(in_features=512, out_features=512, bias=True)
        )
        (drop_out): Dropout(p=0.1, inplace=False)
        (layer_norm2): LayerNormalization(
          (layer): LayerNorm((512,), eps=1e-06, elementwise_affine=True)
        )
        (feed_forward): FeedForwardLayer(
  

In [ ]:
# 더미 데이터 실행 - 랜덤 토큰 id를 만든다.
src_input = torch.randint(0,src_vocab_size,(batch_size,seq_len)).to(device)
trg_input = torch.randint(0,trg_vocab_size,(batch_size,seq_len)).to(device)

# 학습이 아니라 단순이 실행
model.eval()
with torch.no_grad():
    output = model(src_input,trg_input)

print('src 입력 : ', src_input.shape)
print('trg 입력 : ', trg_input.shape)
print('output : ', output.shape)
# 64개의 문자에 대해 각 문장 128개의 위치마다 1000개의 단어 중 무엇이 나올지 점수가 출력되고 있다.

src 입력 :  torch.Size([64, 128])
trg 입력 :  torch.Size([64, 128])
output :  torch.Size([64, 128, 10000])
